# Create xyz and CIF files of Platonic NPs of different sizes and forms from a dataset of compounds (CIF dataset)

## Imports

In [1]:
import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path
import numpy as np
import visualID as vID
from visualID import  fg, hl, bg


from pyNanoMatBuilder import platonicNPs as pNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib

from ase import io
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase.io import read
from ase.io import write
from ase.visualize import view

from debyecalculator import DebyeCalculator
import pyNanoMatBuilder.utilsDC as pyNMBuDC

/home/sara/pyNanoMatBuilder


## Definition of the Platonic Files class

#### NB: the Platonics class is instantiated by defining a compound and its interatomic distance. The function FindInterAtomicDist, which is defined in utils, calculates the interatomic distance by extracting the cell paratemers and uses well known formulas for fcc, bcc and hcp. Thus, compounds with unknown interatomic distances are not used to create Platonic NPs.

<span style="color:#FF0000"> Problem : not possible to create the forms for multiple elements like TiO2 or NaCl


In [2]:
class PlatonicFiles :
    """
    A class for generating XYZ and CIF files of Platonic nanoparticles (NPs) 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,form: str=None,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data, shapes and sizes.
            Args:
        path (str) : Path that will contain the created xyz files
        cif data (DataFrame):  DataFrame containing CIF files of different compounds
        sizes (array-like): Array of the number of bonds per edges, ex: [[1],[2],[3]]
        form (str, optional): If None, all the forms are created; otherwise, a specific form is selected.
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): If False, details of the files are printed.
            Methods:
        create_platonic(noOutput, path): Generates Platonic nanoparticles and saves their files.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # Dictionary to store loaded CIF files
        self.sizes= sizes # List of sizes 
        self.form= form # Specific form to generate (if any)
        
        self.create_platonic(noOutput,max_size,path) # Call method to generate Platonic nanoparticles
  
    def create_platonic(self,noOutput,max_size,path):
        """
        Genenrate platonic NPs and their files from cif data.
        Args:
            noOutput (bool): If False, prints details about the process.
            path (str): Path where xyz/cif files will be created.
        Note:
        - The nanoparticles are created using classes of the module`pNP`:
          - `pNP.regfccOh`
          - `pNP.cube`
          - `pNP.regIco`
          - `pNP.regfccTd`
          - `pNP.regDD`
        """
              
        plat_dict= {
            'regfccOh': 'nOrder',
            'cube': 'nOrder',
            'regIco': 'nShell',
            'regfccTd': 'nLayer',
            'regDD':'nShell'
            }
        
        form_fcc=['regfccOh','regfccTd','regDD','regIco','cube']
        form_bcc=['regDD','regIco','cube']
        form_other=['regDD','regIco']
        
        for cif_name, cif_file in self.cif_data['cif file'].items():
            
            # Load cif informations
            element=cif_name.split()[0] # Extract chemical element from CIF name
            
            if not (element=='TiO2' or element=='NaCl'): # Skip TiO2 and NaCl since they are not processed
                self.cif_name=cif_name
                
                # Extract the structure name for the name of the files, for example 'Rutile' or 'Anatase' or 'Alpha'
                if len(self.cif_name.split())==2 : 
                    structure=self.cif_name.split()[1]
                else : 
                    structure=None
                if not noOutput :
                    print(f'\n\033[1m{cif_name.center(50)}\033[0m\n')
                cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
                    
                # Determine forms based on crystal type
                if self.crystal_type=='fcc' :
                    forms=form_fcc
                    dist= pyNMBu.FindInterAtomicDist(self)# Extract the interatomic distance
                elif self.crystal_type=='bcc':
                    forms=form_bcc
                    dist= pyNMBu.FindInterAtomicDist(self) # Extract the interatomic distance
                elif self.crystal_type=='hcp':
                    forms=form_other
                    dist= pyNMBu.FindInterAtomicDist(self) # Extract the interatomic distance
                else : 
                    forms=None
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} xyz/cif files can't be created for {self.cif_name} because the interatomic distance is unknown.\033[0m  ")   
                        
                if not noOutput :
                    print(f" \033[1m The forms possible of {self.cif_name}({self.crystal_type}) are {forms}.\033[0m  ") 
                
                
                if self.form == None :  # If no specific form is selected, generate all possible forms
                    if not forms==None :
       
                        for form in forms:
                            n_size=plat_dict[form]
                            cls=getattr(pNP,form)  # Get the class corresponding to the form (pNP.form)
                            index=0 
                            if not noOutput :
                                print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type} for the form {form}. \033[0m ")
                            if n_size=='nLayer' :
                                # Create instances for each form and size
                                for i in self.sizes :
                                    index += 1 
                                    if not noOutput :
                                        print(f'Number of bonds is {i}')
                                    TestNP =cls(
                                        element=element,
                                        Rnn=dist,
                                        **{n_size:i+1},
                                        shape=f'{form}',
                                        postAnalyzis=True,
                                        aseView=False,
                                        thresholdCoreSurface=1,
                                        skipSymmetryAnalyzis=True,
                                        noOutput= True
                                    )
                                    circumsphere_diameter=TestNP.radiusCircumscribedSphere()*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                                    if circumsphere_diameter<max_size :
                                        if not noOutput :
                                            print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                                        pyNMBu.writexyz_generalized_platonic(self,structure,path,TestNP,index,noOutput) 
                                    else :
                                        break
                            else :
                                # Create instances for each form and size
                                for i in self.sizes :
                                    index += 1 
                                    if not noOutput :
                                        print(f'Number of bonds is {i}')
                                    TestNP =cls(
                                        element=element,
                                        Rnn=dist,
                                        **{n_size:i},
                                        shape=f'{form}',
                                        postAnalyzis=True,
                                        aseView=False,
                                        thresholdCoreSurface=1,
                                        skipSymmetryAnalyzis=True,
                                        noOutput= True
                                    )
                                    circumsphere_diameter=TestNP.radiusCircumscribedSphere()*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                                    if circumsphere_diameter<max_size :
                                        if not noOutput :
                                            print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                                        pyNMBu.writexyz_generalized_platonic(self,structure,path,TestNP,index,noOutput)               
                                    else :
                                        break
          
                else: # If form isnt None = if a form is given by the user
                    if not forms==None :
                        if f'{self.form}' in forms:
                            n_size=plat_dict[self.form]
                            cls=getattr(pNP,self.form)  # Get the class corresponding to the form (pNP.form)
                            index=0 
                            if not noOutput :
                                print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type} for the form {self.form}. \033[0m ")
                            if n_size=='nLayer' :
                                # Create instances for each form and size
                                for i in self.sizes :
                                    index += 1 
                                    if not noOutput :
                                        print(f' {bg.LIGHTBLUEB} Number of bonds is {i}')
                                    TestNP =cls(
                                        element=element,
                                        Rnn=dist,
                                        **{n_size:i+1},
                                        shape=f'{self.form}',
                                        postAnalyzis=True,
                                        aseView=False,
                                        thresholdCoreSurface=1,
                                        skipSymmetryAnalyzis=True,
                                        noOutput= True
                                    )   
                                    circumsphere_diameter=TestNP.radiusCircumscribedSphere()*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                                    if circumsphere_diameter<max_size :
                                        if not noOutput :
                                            print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter} \033[0m')
                                        pyNMBu.writexyz_generalized_platonic(self,structure,path,TestNP,index,noOutput)               
                                    else :
                                        break
                                        
                            else :
                                # Create instances for each form and size
                                for i in self.sizes :
                                    index += 1 
                                    if not noOutput :
                                        print(f'{bg.LIGHTBLUEB} Number of bonds is {i}')
                                    TestNP =cls(
                                        element=element,
                                        Rnn=dist,
                                        **{n_size:i},
                                        shape=f'{self.form}',
                                        postAnalyzis=True,
                                        aseView=False,
                                        thresholdCoreSurface=1,
                                        skipSymmetryAnalyzis=True,
                                        noOutput= True
                                    )
                                    
                                    circumsphere_diameter=TestNP.radiusCircumscribedSphere()*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                                    if circumsphere_diameter<max_size :
                                        if not noOutput :
                                            print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                                        pyNMBu.writexyz_generalized_platonic(self,structure,path,TestNP,index,noOutput)               
                                    else :
                                        break

            else :
                pass

In [3]:
PlatonicFiles?

Init signature:
PlatonicFiles(
    path,
    cif_data,
    sizes,
    form: str = None,
    max_size: float = 50,
    noOutput: bool = True,
)
Docstring:     
A class for generating XYZ and CIF files of Platonic nanoparticles (NPs) 
from a dataset of compounds (CIF dataset). This process enables the creation 
of a well-structured database optimized for machine learning applications, 
ensuring consistency in format and data representation.
Init docstring:
Initialize the class with CIF data, shapes and sizes.
    Args:
path (str) : Path that will contain the created xyz files
cif data (DataFrame):  DataFrame containing CIF files of different compounds
sizes (array-like): Array of the number of bonds per edges, ex: [[1],[2],[3]]
form (str, optional): If None, all the forms are created; otherwise, a specific form is selected.
max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
noOutput (bool): If False, details

### Example of use of the PlatonicFiles class

##### 1) Create all the possible forms from a dataset of CIF files

If noOutput=False :  
 - <span style="color:#008000"> in green : the allowed shapes created (interatomic distance known) </span> 
 - <span style="color:#FF0000">in red : the shapes that are not created (interatomic distance unknown)</span>

In [4]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/platonic_files_1_5nm'

# Sizes of the NPs
sizes=np.arange(1,25,1) # Control the sizes by the number of bonds per edges : here 1 to 25 bonds (step of 1)

# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the maximum number of bonds defined, here 25.
class_test=PlatonicFiles(path,cif_data=data.pyNMBcif.CIFdf,sizes=sizes,max_size=5,noOutput= False) # Ex: the maximal size is a circumsphere diameter of 5 nm.



print(t.chrono_stop(hdelay=True))


                      Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
  The forms possible of Fe bcc(bcc) are ['regDD', 'regIco', 'cube'].  
  xyz/cif files can be created for Fe bcc of Bravais lattice=bcc for the form regDD.  
Number of bonds is 1
Circumscribed sphere diameter =0.6943307452523548
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000001_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000001_0000000.cif
Number of bonds is 2
Circumscribed sphere diameter =1.3886614905047097
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000002_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000002_0000000.cif
Number of bonds is 3


/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


Circumscribed sphere diameter =2.082992235757064
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000003_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000003_0000000.cif
Number of bonds is 4
Circumscribed sphere diameter =2.7773229810094193
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000004_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000004_0000000.cif
Number of bonds is 5
Circumscribed sphere diameter =3.4716537262617737
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000005_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000005_0000000.cif
Number of bonds is 6
Circumscribed sphere diameter =4.165984471514128
  xyz file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000006_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Fe_bcc_regDD_0000006_0000000.cif
Number of bonds is 7
Circumscri

##### 2) Create a specific form from a dataset of CIF files (ex: regfccTd)

In [5]:
t = pyNMBu.timer()
t.chrono_start()

path='database_1_5nm/platonic_files_1_5nm'

sizes=np.arange(1,17,1) # sizes : 1 to 17 bonds

class_test=PlatonicFiles(path,cif_data=data.pyNMBcif.CIFdf,sizes=sizes,form='regfccTd',max_size=5,noOutput= False)

print(t.chrono_stop(hdelay=True))


                      Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
  The forms possible of Fe bcc(bcc) are ['regDD', 'regIco', 'cube'].  

                     Mn alpha                     

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9011068-Mn_alpha.cif
  xyz/cif files can't be created for Mn alpha because the interatomic distance is unknown.  
  The forms possible of Mn alpha(cubic) are None.  

                     Mn beta                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1539039-Mn_beta.cif
  xyz/cif files can't be created for Mn beta because the interatomic distance is unknown.  
  The forms possible of Mn beta(cubic) are None.  

                      Co hcp                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9008492-Co_hcp.cif
  The forms possible of Co hcp(hcp) are ['regDD', 'regIco'].  

                      

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


Circumscribed sphere diameter =3.6871897591526266 
  xyz file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000012_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000012_0000000.cif
  Number of bonds is 13
Circumscribed sphere diameter =3.9944555724153448 
  xyz file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000013_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000013_0000000.cif
  Number of bonds is 14
Circumscribed sphere diameter =4.301721385678064 
  xyz file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000014_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000014_0000000.cif
  Number of bonds is 15
Circumscribed sphere diameter =4.608987198940783 
  xyz file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000015_0000000.xyz 
  cif file created:database_1_5nm/platonic_files_1_5nm/Co_fcc_regfccTd_0000015_000000